# Open-ended massive-halo (log10 M_200c > 12) satellite quenching anisotropy — TNG100, z = 0

Companion to `03_analysis` / `08`-`09`, for **all hosts with log10 M_200c [phys M_sun] > 12.0**
(lower bound only, **no upper limit**). Self-contained: it (1) rebuilds a per-satellite catalog from
the raw TNG100 data with that open host cut, (2) keeps only satellites **within 1 R_200c** of their
host, (3) writes a CSV whose name carries the cut (`hostlogM12.0plus`) so it never overwrites another
catalog, and (4) repeats the same `a + b*cos(2 theta)` quench-fraction **MCMC** used for the
lower-mass centrals.

Snapshot 099 (z = 0.0). Catalog-build half from `01_tng_generate_catalogs`;
MCMC half from `03_analysis`. Requires the raw TNG100 data under `~/SimulationData/L75n1820TNG` and
`illustris_python` (run on the cluster/Binder, as with notebook 01).

> Only the **host mass cut** (open: > 12.0) and the **output filename** differ from the MW-mass
> pipeline; the satellite-mass grid, R_200c cut, angle/quench definitions and MCMC are unchanged.
> SAGA/ELVES are dropped — they are MW-mass observational samples with no high-mass counterpart.

# Part 1 — build the satellite catalog

## Imports

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import h5py
import illustris_python as il
from scipy.stats import binned_statistic

## Configuration

In [ ]:
# --- simulation + redshift (fixed for this notebook) ---
SIM      = 'TNG100'
REDSHIFT = 'z0'                  # this notebook: z = 0

_SIM_DIR    = {'TNG100': 'L75n1820TNG', 'TNG50': 'L35n2160TNG'}[SIM]
_LBOX_MPC_H = {'TNG100': 75.0,         'TNG50': 35.0}[SIM]
snap        = {'z0': 99, 'z0p05': 95}[REDSHIFT]      # z=0 -> 99, z=0.05 -> 95 (project convention)
_Z          = {'z0': 0.0, 'z0p05': 0.0485}[REDSHIFT]
_A          = 1.0 / (1.0 + _Z)                        # comoving -> physical (a=1 at z=0)

TNG_ROOT = os.path.expanduser(f'~/SimulationData/{_SIM_DIR}')
_sp             = f'snapnum_{snap:03d}'
basePath        = os.path.join(TNG_ROOT, 'output')
morph_g         = os.path.join(TNG_ROOT, f'postprocessing/skirt_images/sdss/{_sp}/morphs_g.hdf5')
morph_i         = os.path.join(TNG_ROOT, f'postprocessing/skirt_images/sdss/{_sp}/morphs_i.hdf5')
subfind_id_path = os.path.join(TNG_ROOT, f'postprocessing/skirt_images/sdss/{_sp}/subfind_ids.txt')

_gc = os.path.join(basePath, f'groups_{snap:03d}', f'fof_subhalo_tab_{snap:03d}.0.hdf5')
assert os.path.isdir(basePath), f'basePath does not exist: {basePath}  -> edit TNG_ROOT.'
assert os.path.exists(_gc),     f'group catalog not found: {_gc}'
assert os.path.exists(morph_g), f'SDSS morphology not found: {morph_g}'
print(f'SIM = {SIM} | REDSHIFT = {REDSHIFT} (snap {snap}) | TNG_ROOT = {TNG_ROOT}')

# --- cosmology / box ---
h    = 0.6774
Lbox = _LBOX_MPC_H * 1000.0 / h                       # box side in physical kpc

# --- selections: OPEN host cut (lower bound only, no upper limit) ---
HOST_LOGM200_MIN = 12.0            # log10 M_200c [physical M_sun]; everything above is kept
HOST_LOGM200_MAX = None                  # no upper limit
SAT_MASS_CUTS    = 10.0 ** np.array([6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0])  # same grid as notebook 01
HOST_TAG         = f'hostlogM{HOST_LOGM200_MIN:.1f}plus'   # -> file name tag (open-ended)

SN_MIN = 2.5                            # host SDSS-morphology S/N quality cut

# >>> exclude systems beyond 1 R_200c <<<
WITHIN_R200C = '3d'                    # '3d' = 3D separation, 'proj' = projected, None = no cut
R200C_FACTOR = 1.0

OUTDIR = f'../data/{SIM.lower()}/{REDSHIFT}'
os.makedirs(OUTDIR, exist_ok=True)

## Load the group / subhalo catalogs and SDSS-SKIRT morphologies

In [ ]:
groups = il.groupcat.loadHalos(
    basePath, snap,
    fields=['GroupFirstSub', 'Group_M_Crit200', 'Group_R_Crit200', 'GroupNsubs'])

subhalos = il.groupcat.loadSubhalos(
    basePath, snap,
    fields=['SubhaloGrNr', 'SubhaloMassType', 'SubhaloCM', 'SubhaloSFR'])

n_sub = len(subhalos['SubhaloGrNr'])

sdss_g = h5py.File(morph_g, 'r')
sdss_i = h5py.File(morph_i, 'r')
subfind_ids = np.loadtxt(subfind_id_path)

print('subhalos:', n_sub, '| groups:', len(groups['GroupFirstSub']))

## Select hosts with log10 M_200c > 12.0 and flag their satellites

`Group_M_Crit200` is in `10^10 h^-1 M_sun`; physical host mass is `*1e10/h`. Open cut: only the lower
bound is applied (`HOST_LOGM200_MAX is None`). Satellites of a group are subhalos
`GroupFirstSub+1 ... GroupFirstSub+GroupNsubs-1` (index `GroupFirstSub` is the central).

In [ ]:
with np.errstate(divide='ignore'):
    host_logm200_phys = np.log10(groups['Group_M_Crit200'] * 1e10 / h)
host_sel = host_logm200_phys > HOST_LOGM200_MIN
if HOST_LOGM200_MAX is not None:                       # open-ended here, but keep the machinery general
    host_sel &= host_logm200_phys < HOST_LOGM200_MAX

first_sub = groups['GroupFirstSub'][host_sel]
n_subs    = groups['GroupNsubs'][host_sel]
print(f'hosts (log10 M200c_phys > {HOST_LOGM200_MIN}):', host_sel.sum())

sat_mask        = np.zeros(n_sub, dtype=bool)
host_id_arr     = np.full(n_sub, -1, dtype=int)
host_center     = np.zeros((n_sub, 3))
host_r200_arr   = np.zeros(n_sub)
host_m200_phys  = np.zeros(n_sub)
host_m200_hinv  = np.zeros(n_sub)

host_m200_hinv_all = np.log10(groups['Group_M_Crit200'][host_sel] * 1e10)
host_m200_phys_all = host_logm200_phys[host_sel]

for k in range(len(first_sub)):
    c   = first_sub[k]
    sl  = slice(c + 1, c + n_subs[k])
    sat_mask[sl]       = True
    host_id_arr[sl]    = k
    host_center[sl]    = subhalos['SubhaloCM'][c] / h
    host_r200_arr[sl]  = groups['Group_R_Crit200'][host_sel][k] / h
    host_m200_phys[sl] = host_m200_phys_all[k]
    host_m200_hinv[sl] = host_m200_hinv_all[k]

print('satellite subhalos flagged:', sat_mask.sum())

## Propagate host Sersic orientation + quality flags to satellites

Host major-axis angle = mean of the SDSS g/i 2D Sersic position angles. We require reliable g+i
fits (`flag == 0`, `flag_sersic == 0`, `sn_per_pixel > SN_MIN`).

> Heads-up: the SDSS-SKIRT morphology sample may not include every massive central. The print below
> reports how many satellites survive the morphology requirement — if it is tiny, relax `SN_MIN` or
> revisit the orientation source.

In [ ]:
def host_morph_to_satellites(sdss, first_sub, n_subs, subfind_ids, n_sub):
    '''Look up each selected host's Sersic morphology and broadcast it to its satellites.'''
    theta = np.zeros(n_sub); flag = np.zeros(n_sub); sflag = np.zeros(n_sub); sn = np.zeros(n_sub)
    theta_all = np.asarray(sdss['sersic_theta']) * 180.0 / np.pi
    flag_all  = np.asarray(sdss['flag'])
    sflag_all = np.asarray(sdss['flag_sersic'])
    sn_all    = np.asarray(sdss['sn_per_pixel'])
    for k in range(len(first_sub)):
        c   = first_sub[k]
        row = np.where(subfind_ids == c)[0]
        if row.size == 0:
            flag[c + 1:c + n_subs[k]] = 1
            continue
        r  = row[0]
        sl = slice(c + 1, c + n_subs[k])
        theta[sl] = theta_all[r]; flag[sl] = flag_all[r]
        sflag[sl] = sflag_all[r]; sn[sl] = sn_all[r]
    return theta, flag, sflag, sn

theta_g, flag_g, sflag_g, sn_g = host_morph_to_satellites(sdss_g, first_sub, n_subs, subfind_ids, n_sub)
theta_i, flag_i, sflag_i, sn_i = host_morph_to_satellites(sdss_i, first_sub, n_subs, subfind_ids, n_sub)

host_theta = 0.5 * (theta_g + theta_i)
host_good  = ((flag_g == 0) & (sflag_g == 0) & (sn_g > SN_MIN) &
              (flag_i == 0) & (sflag_i == 0) & (sn_i > SN_MIN))
print('satellites whose host has reliable g+i Sersic fits:', (sat_mask & host_good).sum())

## Projected azimuthal angle alpha (0 deg = major axis, 90 deg = minor axis)

In [ ]:
def wrap_min_image(d, L):
    return d - L * np.round(d / L)

def angle_from_major_axis(phi_sat_deg, pa_host_deg):
    '''Fold |phi_sat - PA_host| into [0, 90] using the disk's 180-deg / mirror symmetry.'''
    phi = np.mod(phi_sat_deg, 360.0)
    d1 = np.abs(phi - np.mod(pa_host_deg, 360.0))
    d2 = np.abs(phi - np.mod(pa_host_deg + 180.0, 360.0))
    d1 = np.minimum(d1, 360.0 - d1); d2 = np.minimum(d2, 360.0 - d2)
    m  = np.minimum(d1, d2)
    return np.minimum(m, 180.0 - m)

rel    = wrap_min_image(subhalos['SubhaloCM'] / h - host_center, Lbox)
phi_2d = np.degrees(np.arctan2(rel[:, 1], rel[:, 0]))
alpha  = angle_from_major_axis(phi_2d, host_theta)

_d3d_com   = np.linalg.norm(rel, axis=1)
_dproj_com = np.hypot(rel[:, 0], rel[:, 1])
d_3d_kpc   = _d3d_com   * _A
d_proj_kpc = _dproj_com * _A
with np.errstate(invalid='ignore', divide='ignore'):
    _good_r     = host_r200_arr > 0
    d_r200_3d   = np.where(_good_r, _d3d_com   / host_r200_arr, np.nan)
    d_r200_proj = np.where(_good_r, _dproj_com / host_r200_arr, np.nan)

## Satellite stellar mass, SFR, quenched flag

Quenched = >= 1 dex below the SDSS SFMS of Martin-Navarro+ 2021,
`log10(SFR_MS) = 0.75*log10(M*) - 7.5` (physical M*).

In [ ]:
with np.errstate(divide='ignore'):
    mstar_phys = np.log10(subhalos['SubhaloMassType'][:, 4] * 1e10 / h)
    mstar_hinv = np.log10(subhalos['SubhaloMassType'][:, 4] * 1e10)
sat_sfr = np.asarray(subhalos['SubhaloSFR'])

def quenched_flag(logmstar_phys, sfr):
    sfr_ms = 10.0 ** (0.75 * logmstar_phys - 7.5)
    return (sfr < sfr_ms / 10.0).astype(int)

## Build and write one catalog per satellite mass cut

Selection per cut: satellite of a host with `log10 M_200c > 12.0` **and reliable g+i Sersic
fits**, **within 1 R_200c** (3D), and `M*,sat (phys) > cut`. Files are written as
`tng100_satellites_{HOST_TAG}_logM*.csv` so they sit alongside — and never overwrite — the other
catalogs.

In [ ]:
def cut_tag(cut):
    return f'logM{np.log10(cut):.2f}'

# within-R_200c selection (excludes everything beyond 1 R200c)
if WITHIN_R200C == '3d':
    radius_ok = d_r200_3d < R200C_FACTOR
elif WITHIN_R200C == 'proj':
    radius_ok = d_r200_proj < R200C_FACTOR
elif WITHIN_R200C in (None, 'none'):
    radius_ok = np.ones(len(sat_mask), dtype=bool)
else:
    raise ValueError("WITHIN_R200C must be '3d', 'proj', or None")
print(f'within-R200c cut: {WITHIN_R200C} (< {R200C_FACTOR} R200c)')

tags, logcuts, fq_list, nsat_list = [], [], [], []
for cut in SAT_MASS_CUTS:
    sel = sat_mask & host_good & radius_ok & (mstar_phys > np.log10(cut))

    df = pd.DataFrame({
        'host_id':        host_id_arr[sel],
        'mstar_phys':     mstar_phys[sel],
        'mstar_hinv':     mstar_hinv[sel],
        'host_m200_phys': host_m200_phys[sel],
        'host_m200_hinv': host_m200_hinv[sel],
        'sfr':            sat_sfr[sel],
        'alpha':          alpha[sel],
        'd_3d_kpc':       d_3d_kpc[sel],
        'd_proj_kpc':     d_proj_kpc[sel],
        'd_r200_3d':      d_r200_3d[sel],
        'd_r200_proj':    d_r200_proj[sel],
        'quenched':       quenched_flag(mstar_phys[sel], sat_sfr[sel]),
    })

    assert np.allclose(df.mstar_phys - df.mstar_hinv, -np.log10(h), atol=1e-6), 'h-convention mismatch!'
    assert df.alpha.between(0, 90).all(), 'alpha outside [0, 90]!'
    if WITHIN_R200C == '3d':
        assert (df.d_r200_3d < R200C_FACTOR).all(), 'a satellite slipped past the 1 R200c cut!'

    tag = cut_tag(cut)
    tags.append(tag); logcuts.append(np.log10(cut))
    fq_list.append(df.quenched.mean()); nsat_list.append(len(df))
    out = f'{OUTDIR}/tng100_satellites_{HOST_TAG}_{tag}.csv'
    df.to_csv(out, index=False)
    print(f'logM*>{np.log10(cut):.2f}  N_sat={len(df):6d}  N_host={df.host_id.nunique():4d}  '
          f'f_q={df.quenched.mean():.3f}  -> {os.path.basename(out)}')

print(f'\nwrote {len(tags)} catalogs to {OUTDIR}')
if fq_list[0] > fq_list[-1]:
    print('note: f_q decreases from the lowest to the highest satellite mass cut.')
else:
    print('note: f_q does NOT monotonically decrease with satellite mass cut here (not an error).')

# Part 2 — anisotropy + quench-fraction sinusoid MCMC

Same procedure as the lower-mass centrals (`03_analysis`): bin the within-R_200c satellites in
azimuthal angle, bootstrap the per-bin quench fraction, and fit `f_q(theta) = a + b*cos(2 theta)`
with `emcee`. `b > 0` means quenching is enhanced toward the host **major axis**. TNG-only.

In [ ]:
import scipy.stats as stats
import emcee
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
})

# 18 angle bins over [0, 90] (same as notebook 03)
N_BINS = 18
ANGLE_EDGES   = np.linspace(0, 90, N_BINS + 1)
ANGLE_CENTERS = 0.5 * (ANGLE_EDGES[:-1] + ANGLE_EDGES[1:])

TNG_DIR     = OUTDIR
FIG_LOGCUTS = [6, 7, 8]      # satellite log10(M*/Msun) cuts to feature (same as notebook 03)

## Load the catalogs we just wrote

In [ ]:
avail = {}
for p in glob.glob(f'{TNG_DIR}/tng100_satellites_{HOST_TAG}_*.csv'):
    m = re.search(r'logM([0-9.]+)\.csv$', os.path.basename(p))
    if m:
        avail[float(m.group(1))] = p
if not avail:
    raise FileNotFoundError(f'no tng100_satellites_{HOST_TAG}_*.csv in {TNG_DIR} -- run Part 1 first.')

tng = {}
for lc in FIG_LOGCUTS:
    nearest = min(avail, key=lambda a: abs(a - lc))
    tng[f'1e{lc}'] = pd.read_csv(avail[nearest])
    print(f'requested logM>{lc}: using {os.path.basename(avail[nearest])} '
          f'(logM={nearest:.2f}) | {len(tng[f"1e{lc}"])} sats, f_q={tng[f"1e{lc}"].quenched.mean():.3f}')

## Normalized satellite count vs azimuthal angle

Excess near 0 deg = alignment with the host major axis; flat = isotropic.

In [ ]:
def norm_hist(angles):
    c, _ = np.histogram(angles, bins=ANGLE_EDGES)
    return c / c.sum() / (90.0 / N_BINS)

fig, ax = plt.subplots(figsize=(7, 5))
for cut, ls in zip(list(tng), [':', '--', '-']):
    h = norm_hist(tng[cut]['alpha'])
    ax.step(ANGLE_EDGES, np.r_[h, h[-1]], where='post', color='k', ls=ls, lw=2,
            label=r'$M_{*,\rm sat}>10^{%d}\,M_\odot$' % int(np.log10(float(cut))))
ax.set_xlim(0, 90); ax.set_ylim(0, 0.030)
ax.set_xlabel(r'$\theta$ [deg]'); ax.set_ylabel(r'P($\theta$)')
ax.tick_params(which='both', direction='in', top=True, right=True)
ax.legend(title=f'TNG100 {HOST_TAG}:', fancybox=False, edgecolor='k')
plt.show()

## KS tests

In [ ]:
print('One-sample KS vs Uniform(0, 90):')
for cut in list(tng):
    print(f'  TNG {cut}: {stats.kstest(tng[cut]["alpha"].to_numpy(), "uniform", args=(0, 90))}')

print('\nTwo-sample KS, quenched vs star-forming azimuthal distribution:')
for cut in list(tng):
    d = tng[cut]
    print(f'  TNG {cut}: {stats.ks_2samp(d.alpha[d.quenched == 1], d.alpha[d.quenched == 0])}')

## Quench fraction vs angle: bootstrap + sinusoid MCMC fit

Model `f_q(theta) = a + b*cos(2 theta)` with a Gaussian jitter term `exp(f)`.

In [ ]:
def bootstrap_fq(angle, quenched, N=10000, seed=0):
    '''Binned-mean quench fraction per angle bin, with bootstrap mean and 1-sigma error.'''
    rng = np.random.default_rng(seed)
    angle = np.asarray(angle); quenched = np.asarray(quenched, dtype=float)
    n = len(angle)
    boot = np.full((N, N_BINS), np.nan)
    bin_idx = np.digitize(angle, ANGLE_EDGES) - 1
    for i in range(N):
        s = rng.integers(0, n, n)
        bi, qi = bin_idx[s], quenched[s]
        for j in range(N_BINS):
            mm = bi == j
            if mm.any():
                boot[i, j] = qi[mm].mean()
    return np.nanmean(boot, axis=0), np.nanstd(boot, axis=0)

fq = {}
for cut in list(tng):
    fq[cut] = bootstrap_fq(tng[cut]['alpha'].to_numpy(), tng[cut]['quenched'].to_numpy())

In [ ]:
def log_likelihood(theta, x, y, sigma):
    a, b, f = theta
    s = sigma ** 2 + np.exp(f) ** 2
    model = a + b * np.cos(2 * np.radians(x))
    return -0.5 * np.sum((y - model) ** 2 / s + np.log(2 * np.pi * s))

def log_prior(theta):
    a, b, f = theta
    return 0.0 if (0 < a < 1 and -1 < b < 1 and -10 < f < 2) else -np.inf

def log_prob(theta, x, y, sigma):
    lp = log_prior(theta)
    return lp + log_likelihood(theta, x, y, sigma) if np.isfinite(lp) else -np.inf

def fit_sinusoid(mean, std, n_walkers=20, n_steps=10000, burn=1000, seed=0):
    '''MCMC fit of a + b*cos(2 theta); returns (params_mean, params_std).'''
    np.random.seed(seed)
    ok = np.isfinite(mean) & np.isfinite(std) & (std > 0)
    p0 = np.array([0.7, 0.025, -3.0]) + 1e-2 * np.random.randn(n_walkers, 3)
    sampler = emcee.EnsembleSampler(n_walkers, 3, log_prob,
                                    args=(ANGLE_CENTERS[ok], mean[ok], std[ok]))
    sampler.run_mcmc(p0, n_steps, progress=False)
    chain = sampler.get_chain(discard=burn, flat=True)
    return chain.mean(axis=0), chain.std(axis=0)

params = {k: fit_sinusoid(*fq[k]) for k in fq}
for k, (p, e) in params.items():
    print(f'{k:>5}:  a = {p[0]:.3f} +/- {e[0]:.3f}   b = {p[1]:+.3f} +/- {e[1]:.3f}   '
          f'|b|/sigma_b = {abs(p[1] / e[1]):.2f}')

## Quench fraction vs angle with sinusoid fits

Shaded bands are the 16-84th percentile of the sinusoid model from the MCMC posterior.

In [ ]:
def model_band(p, e, n_mc=10000, seed=0):
    rng = np.random.default_rng(seed)
    x = np.linspace(0, np.pi / 2, 400)
    a = rng.normal(p[0], e[0], n_mc); b = rng.normal(p[1], e[1], n_mc)
    y = a[:, None] + b[:, None] * np.cos(2 * x)[None, :]
    return np.degrees(x), (a.mean() + b.mean() * np.cos(2 * x)), np.percentile(y, 16, 0), np.percentile(y, 84, 0)

fig, ax = plt.subplots(figsize=(7, 5))
for cut, ls in zip(list(tng), [':', '--', '-']):
    m, s = fq[cut]
    ax.errorbar(ANGLE_CENTERS, m, yerr=s, fmt='o', color='k', capsize=3)
    xdeg, ymed, ylo, yhi = model_band(*params[cut])
    ax.plot(xdeg, ymed, color='k', lw=2, ls=ls,
            label=r'$M_{*,\rm sat}>10^{%d}\,M_\odot$' % int(np.log10(float(cut))))
    ax.fill_between(xdeg, ylo, yhi, color='k', alpha=0.1)
ax.set_xlim(0, 90); ax.set_ylim(0, 1)
ax.set_xlabel(r'$\theta$ [deg]'); ax.set_ylabel(r'$f_q$')
ax.tick_params(which='both', direction='in', top=True, right=True)
ax.legend(title=f'TNG100 {HOST_TAG}:', fancybox=False, edgecolor='k')
plt.show()

## Sinusoid vs constant: BIC / AIC and amplitude significance

Is the sinusoid a better description than a constant? `Delta(BIC) > 10` is strong evidence.

In [ ]:
def info_criteria(x, y, yerr, p):
    ok = np.isfinite(y) & np.isfinite(yerr) & (yerr > 0)
    x, y, yerr = x[ok], y[ok], yerr[ok]
    n = len(y)
    a, b, f = p
    resid_sin = y - (a + b * np.cos(2 * np.radians(x)))
    logL_sin  = -0.5 * np.sum((resid_sin / yerr) ** 2 + np.log(2 * np.pi * yerr ** 2))
    resid_con = y - np.mean(y)
    logL_con  = -0.5 * np.sum((resid_con / yerr) ** 2 + np.log(2 * np.pi * yerr ** 2))
    bic_sin = 2 * np.log(n) - 2 * logL_sin;  aic_sin = 2 * 2 - 2 * logL_sin
    bic_con = 1 * np.log(n) - 2 * logL_con;  aic_con = 2 * 1 - 2 * logL_con
    return bic_sin, bic_con, aic_sin, aic_con

for key in list(tng):
    m, s = fq[key]; p, e = params[key]
    bs, bc, as_, ac = info_criteria(ANGLE_CENTERS, m, s, p)
    pref_bic = 'sinusoid' if bs < bc else 'constant'
    print(f'{key:>5}: dBIC = {abs(bs - bc):5.2f} ({pref_bic} preferred)  '
          f'dAIC = {abs(as_ - ac):5.2f}   |b|/sigma_b = {abs(p[1] / e[1]):.2f}')

## Alignment effect size  <cos 2 theta>

`<cos 2 theta> > 0` means an excess near the major axis; 95% CI from bootstrap.

In [ ]:
def cos2theta_ci(theta_deg, B=5000, seed=0):
    rng = np.random.default_rng(seed)
    t = np.radians(np.asarray(theta_deg))
    A = np.mean(np.cos(2 * t))
    boot = [np.mean(np.cos(2 * rng.choice(t, size=len(t), replace=True))) for _ in range(B)]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return A, lo, hi

for cut in list(tng):
    A, lo, hi = cos2theta_ci(tng[cut]['alpha'])
    print(f'TNG {cut:>4}: <cos 2theta> = {A:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]')